# `mcp` Buyer Notebook — the x402-over-MCP workflow, end to end (Deno/TypeScript)

Drives the **real, locally-running** `mcp_server.ts` (`cd scw_js && npm run dev:mcp`, port 8086)
as the official client stack would: the MCP TypeScript SDK's `StreamableHTTPClientTransport`
against our hand-rolled stateless dispatcher, wrapped by `@x402/mcp`'s
`createx402MCPClient` for payment. No custom protocol code anywhere — if a step needs a
workaround, that is a spike finding, not a notebook bug.

This is the buyer-side spike for the Task 5 checklist, and it answers six open questions
the unit tests mock away:

1. Does the **official MCP client** accept our stateless-JSON server (405 on GET/DELETE, no sessions)?
2. Is our payment-required result actually the **x402-over-MCP convention** — does the official
   wrapper extract the demands and pay without custom code?
3. Do **batch-settlement channel deposits** work through the MCP wrapper (first paid call)?
4. Does the **EURe Permit2** path survive the MCP challenge (the permit rides in the challenge extensions)?
5. Does `llm_complete` settle **usage-derived**, not the advertised ceiling (the settlement-parity fix)?
6. (Deployed run) Does the **Scaleway gateway** pass POST/202/405 cleanly — flip `USE_DEPLOYED` below.

**Prerequisites:** the Deno kernel; `npm run dev:mcp` running; `TEST_WALLET_PRIVATE_KEY` in
`scw_js/.env` funded with Base Sepolia USDC, EURC and EURe (plus a little ETH for the channel
deposit's gas). Same wallet/channel state as the other buyer notebooks — see the channel-reuse
note below.

In [1]:
// Setup: imports + env
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";

// scw_js's own .env, one level up — the single source of truth (buyer key, receiver
// address, and the server's own secrets all live here; the server reads the same file).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error(
        "TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with Base Sepolia " +
        "USDC, EURC and EURe, plus a little ETH for the channel deposit's gas.",
    );
}
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("mcp_x402 buyer notebook");
console.log(`   Buyer (payer): ${account.address}`);
console.log(`   Receiver     : ${env.NFT_WALLET_PUBLIC_KEY ?? "(not set — server will 500)"}`);

mcp_x402 buyer notebook
   Buyer (payer): 0x553179556FC2A39e535D65b921e01fA995E79101
   Receiver     : 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Network selection

Two flags, same meaning as the other buyer notebooks. The MCP tools offer the same
per-currency batch-settlement terms as the REST twins (`mcp_tools.ts` builds its accepts from
the very same `createBatchSettlementPaymentRequirements`), so everything the
`sc_llm_x402_buyer` README says about channels applies unchanged.

In [2]:
import { base, baseSepolia, optimism } from "npm:viem@2/chains";

const USE_MAINNET = false;  // true = REAL MONEY: mainnet settlement + billed Brave/Mistral calls.
const USE_BASE = true;       // Optimism is mainnet-only for batch-settlement, see guard below.

if (!USE_BASE && !USE_MAINNET) {
    throw new Error(
        "Optimism Sepolia is unsupported for batch-settlement (no canonical contract). " +
        "Use Base Sepolia (USE_BASE = true) or an Optimism mainnet run (USE_MAINNET = true).",
    );
}

const NETWORK_CONFIG = {
    "base-testnet": {
        caip2Network: "eip155:84532" as const, networkName: "Base Sepolia (Testnet)",
        chain: baseSepolia, usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        // EURC on Base Sepolia — shared/chain-utils/src/addresses.ts and scw_js/README.md's
        // EIP-712 domain-name table.
        eurcAddress: "0x808456652fdb597867f38412077A9182bf77359F" as `0x${string}` | undefined,
        // EURe (Monerium, 18 decimals, Permit2 + EIP-2612 permit — no approve(), no ETH for approval).
        eureAddress: "0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85" as `0x${string}` | undefined,
        rpcUrl: undefined as string | undefined,
        explorer: "https://sepolia.basescan.org", faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        caip2Network: "eip155:8453" as const, networkName: "Base Mainnet",
        chain: base, usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        eurcAddress: "0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42" as `0x${string}` | undefined,
        eureAddress: "0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C" as `0x${string}` | undefined,
        rpcUrl: "https://base-rpc.publicnode.com",
        explorer: "https://basescan.org", faucet: "Bridge: https://bridge.base.org",
    },
    "optimism-mainnet": {
        caip2Network: "eip155:10" as const, networkName: "Optimism Mainnet",
        chain: optimism, usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        // No EURC on Optimism (Circle has no deployment there).
        eurcAddress: undefined as `0x${string}` | undefined,
        eureAddress: undefined as `0x${string}` | undefined,
        rpcUrl: "https://optimism-rpc.publicnode.com",
        explorer: "https://optimistic.etherscan.io", faucet: "Bridge: https://app.optimism.io/bridge",
    },
};
const configKey = USE_MAINNET ? "base-mainnet" : "base-testnet";
const config = NETWORK_CONFIG[configKey];
const NETWORK = config.caip2Network;

// Local: the dev server (`npm run dev:mcp`, scw_js/dev_server_mcp.ts on 127.0.0.1:8086).
// Deployed: the mcp function's custom domain (serverless.yml). Orthogonal to USE_MAINNET —
// the channel is on-chain state keyed by (buyer, receiver, network, asset), not by which
// server talked to it, so a channel opened here also pays the deployed endpoint, and vice
// versa (same property the search buyer notebook relies on to share the chat's channel).
const USE_DEPLOYED = false;
const MCP_URL = USE_DEPLOYED
    ? "https://mcp-agent.fretchen.eu/mcp"
    : "http://127.0.0.1:8086/mcp";

console.log(USE_MAINNET ? `REAL MONEY on ${config.networkName}` : `${config.networkName} (testnet)`);
console.log(`   ${NETWORK} • USDC ${config.usdcName} @ ${config.usdcAddress}`);
console.log(`   MCP endpoint: ${MCP_URL}`);

Base Sepolia (Testnet) (testnet)
   eip155:84532 • USDC USDC @ 0x036CbD53842c5426634e7929541eC2318f3dCF7e
   MCP endpoint: http://127.0.0.1:8086/mcp


## Buyer setup — the channel scheme, same recipe as the REST buyers

`mcp_tools.ts` verifies and settles through the same batch-settlement stack as
`sc_llm_x402.ts`, so the client side is the recipe `sc_llm_x402_buyer.ipynb` proved:
`toClientEvmSigner` (with a `readContract`-capable public client — the corrective-402
recovery path needs it), `ClientChannelStorage` backed by `localStorage`, and a deposit
floor of $0.50 so one channel covers the session instead of topping up every few calls.

In [3]:
import { toClientEvmSigner } from "npm:@x402/evm@2.28.0";
import {
    BatchSettlementEvmScheme,
    type ClientChannelStorage,
    type BatchSettlementClientContext,
    type BatchSettlementDepositStrategyContext,
} from "npm:@x402/evm@2.28.0/batch-settlement/client";
import { createPublicClient, http } from "npm:viem@2";

const publicClient = createPublicClient({
    chain: config.chain,
    transport: http(config.rpcUrl),
});
const buyerSigner = toClientEvmSigner(
    { address: account.address, signTypedData: (a: unknown) => account.signTypedData(a as never) },
    publicClient,
);

class WebStorageClientChannelStorage implements ClientChannelStorage {
    constructor(private backend: Storage, private prefix = "x402-channel:") {}
    private keyFor(key: string) { return `${this.prefix}${key.toLowerCase()}`; }
    async get(key: string) {
        const raw = this.backend.getItem(this.keyFor(key));
        return raw ? (JSON.parse(raw) as BatchSettlementClientContext) : undefined;
    }
    async set(key: string, context: BatchSettlementClientContext) {
        this.backend.setItem(this.keyFor(key), JSON.stringify(context));
    }
    async delete(key: string) { this.backend.removeItem(this.keyFor(key)); }
}

// Same prefix as the other buyer notebooks: a channel here IS the chat/search channel —
// a browser that has already chatted would pay the MCP tools from the same voucher state.
const buyerStorage = new WebStorageClientChannelStorage(localStorage);

// Floor for channel deposits/top-ups, $0.50 in USDC atomic units — same value and same
// reasoning as useX402Chat.ts's MINIMUM_DEPOSIT_ATOMIC: the SDK default sizes deposits at
// only a few cents (depositMultiplier x per-message ceiling), forcing a real on-chain
// top-up every handful of calls. EURe deposits are the same figure scaled to 18 decimals.
const MINIMUM_DEPOSIT_ATOMIC = 400_000n;

function depositStrategy(context: BatchSettlementDepositStrategyContext): string {
    const required = BigInt(context.minimumDepositAmount);
    return (required > MINIMUM_DEPOSIT_ATOMIC ? required : MINIMUM_DEPOSIT_ATOMIC).toString();
}

const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage, depositStrategy });

// How many channel contexts exist right now — the channel-reuse assertions below count these.
function channelKeyCount(): number {
    let n = 0;
    for (let i = 0; i < localStorage.length; i++) {
        if (localStorage.key(i)!.startsWith("x402-channel:")) n++;
    }
    return n;
}
const channelsBefore = channelKeyCount();
console.log(`   Channel contexts in localStorage: ${channelsBefore}`);

   Channel contexts in localStorage: 7


## The x402 MCP client — two instances, two purposes

Both are the official `createx402MCPClient` (from `@x402/mcp`, the same 2.28.0 line the
server's wrapper comes from) — **no custom client code anywhere**:

- **observer** — `autoPayment: false`: an unpaid `tools/call` must surface the payment demands,
  so we can see exactly what the server's wrapper serves through the official client;
- **payer** — `autoPayment: true` with an `onPaymentRequested` logger: the wrapper must extract
  the demands, open/deposit the channel, pay in `_meta`, retry, and attach the settlement
  receipt. That whole chain is spike questions #2 and #3.

EURC and EURe are not in the SDK's default spend-control registry (USDC is), so both clients
allow-list them — without the entries the client would drop those offers rather than pay.

In [4]:
import { StreamableHTTPClientTransport } from "npm:@modelcontextprotocol/sdk@1.12.1/client/streamableHttp.js";
import { createx402MCPClient } from "npm:@x402/mcp@2.28.0";

function assert(cond: unknown, msg: string): asserts cond {
    if (!cond) throw new Error(`FAILED: ${msg}`);
    console.log(`ok: ${msg}`);
}

const allowedAssets = [
    ...(config.eurcAddress ? [{ network: NETWORK, asset: config.eurcAddress }] : []),
    ...(config.eureAddress ? [{ network: NETWORK, asset: config.eureAddress }] : []),
];

const logDemand = async (ctx: { toolName: string; paymentRequired: { accepts: unknown[] } }) => {
    console.log(`   payment demanded for ${ctx.toolName} (${ctx.paymentRequired.accepts.length} accepts):`);
    for (const a of ctx.paymentRequired.accepts as Array<{ network: string; asset: string; amount: string; payTo: string }>) {
        console.log(`     ${a.network} ${a.asset} amount=${a.amount} payTo=${a.payTo}`);
    }
    return true;
};

// connect() = initialize handshake. Spike question #1 lives here: our server is stateless
// JSON (405 on GET/DELETE); if the official transport chokes on that, it fails right here.
const observer = createx402MCPClient({
    name: "mcp-x402-buyer-notebook", version: "1.0.0",
    schemes: [{ network: NETWORK, client: buyerScheme }],
    spendControls: { allowedAssets },
    autoPayment: false,
});
await observer.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));
console.log("initialize handshake passed (stateless JSON accepted)");

const tools = await observer.listTools();
const toolNames = tools.tools.map((t: { name: string }) => t.name).sort();
console.log(`tools/list: ${JSON.stringify(toolNames)}`);
assert(toolNames.includes("llm_complete"), "server advertises llm_complete");
assert(toolNames.includes("web_search"), "server advertises web_search");

initialize handshake passed (stateless JSON accepted)
tools/list: ["llm_complete","web_search"]
ok: server advertises llm_complete


ok: server advertises web_search


## Unpaid call — the convention check (spike question #2)

`autoPayment: false`, so nothing pays. The client's contract (verified from `@x402/mcp` source):
an unpaid demand never comes back as a plain `callTool` return value — the wrapper extracts the
challenge from our payment-required RESULT and **throws** a payment error (`code: 402`, the
full demands on `err.paymentRequired`). `autoPayment` only controls whether the client pays
before that throw is reached. Reading the demands the clean way is the client's own probe,
`getToolPaymentRequirements` — and the throw itself is the convention round-trip working: had
the client failed to recognize our result, it would have *returned* it as a plain result.

In [5]:
// autoPayment: false — the client must recognize our payment-required RESULT and surface
// it as a thrown payment error (code 402, demands on err.paymentRequired); it never
// returns the demand as a plain callTool result.
let thrown: unknown;
try {
    await observer.callTool("web_search", { q: "ERC-8004 agent registry" });
    throw new Error("expected the unpaid call to throw a payment-required error");
} catch (e) {
    thrown = e;
}

// NOTE: not isPaymentRequiredError() — that guard checks error.data (a JSON-RPC error
// shape), but this thrown client error carries the demands on err.paymentRequired and has
// no data field, so the guard returns false for it. Assert on the fields directly.
assert((thrown as { code?: unknown }).code === 402,
    "unpaid demand throws with code 402 (the client-side form of the payment convention)");
const thrownDemand = (thrown as { paymentRequired?: { accepts: unknown[] } }).paymentRequired;
assert(Array.isArray(thrownDemand?.accepts) && thrownDemand.accepts.length > 0,
    "payment demands ride on err.paymentRequired through the official client");

// The purpose-built probe: same call under the hood, but it returns the demands instead
// of throwing. Its docstring warns it really calls the tool — harmless here, the server's
// unpaid path runs nothing metered (the no-free-path guarantee).
const demand = await observer.getToolPaymentRequirements("web_search", { q: "ERC-8004 agent registry" });
assert(demand !== null, "getToolPaymentRequirements returns the server's demands");
const first = demand!.accepts[0] as { network: string; asset: string; amount: string; payTo: string };
console.log(`   demand[0]: ${first.network} ${first.asset} amount=${first.amount} payTo=${first.payTo}`);
// The order is the server's own preference (EURC before USDC on Base; EURe rides the
// permit2 extension). Whichever is listed first is what the default paid walk below pays.

ok: unpaid demand throws with code 402 (the client-side form of the payment convention)
ok: payment demands ride on err.paymentRequired through the official client
ok: getToolPaymentRequirements returns the server's demands
   demand[0]: eip155:10 0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85 amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Paid walk — first call opens the channel (spike questions #3 and #4)

The payer's first `web_search` must: read the demand, **deposit a real on-chain channel on
Base Sepolia** (a real transaction — this is the big unknown: do channel deposits work through
the MCP wrapper?), pay, retry with the payment in `params._meta["x402/payment"]`, and get the
settlement receipt back. The second call must reuse the same channel — no new deposit, no new
wallet prompt. The payer pins `paymentRequirementsSelector` to USDC: the default selection is
"server-ordered `accepts[0]`", and this server lists mainnet accepts first — on a testnet run
that is `eip155:10`, Optimism **mainnet**. The explicit pin is the guide's own production
advice, and it makes which coin pays deterministic.

In [6]:
const payer = createx402MCPClient({
    name: "mcp-x402-buyer-notebook", version: "1.0.0",
    schemes: [{ network: NETWORK, client: buyerScheme }],
    spendControls: { allowedAssets },
    autoPayment: true,
    onPaymentRequested: logDemand,
    // Explicit selection, per the guide's own production advice: the default is
    // "server-ordered accepts[0]", and this server lists mainnet accepts first — on a
    // testnet run that is eip155:10 (Optimism mainnet USDC). Pin USDC so which coin pays
    // never depends on offer ordering.
    paymentRequirementsSelector: (reqs) =>
        reqs.find((r) => (r.asset ?? "").toLowerCase() === config.usdcAddress.toLowerCase()),
});
await payer.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));

const t0 = performance.now();
const search = await payer.callTool("web_search", { q: "x402 MCP payment protocol" });
console.log(`web_search paid in ${Math.round(performance.now() - t0)}ms (first call: deposit + verify + settle)`);

const searchText = JSON.parse(String(search.content?.[0]?.text ?? "{}"));
console.log(`   results[0]: ${JSON.stringify(searchText.results?.[0]).slice(0, 120)}`);
assert(Array.isArray(searchText.results), "paid web_search returns Brave results");
console.log(`   paymentResponse: ${JSON.stringify(search.paymentResponse)?.slice(0, 220)}`);

const channelsAfterFirst = channelKeyCount();
console.log(`   channel contexts: ${channelsBefore} -> ${channelsAfterFirst}`);

// Second call: same channel, no deposit — the MCP wrapper must reuse the client-side
// channel context exactly like wrapFetchWithPayment does in the REST buyers.
const search2 = await payer.callTool("web_search", { q: "batch settlement payment channels" });
const channelsAfterSecond = channelKeyCount();
assert(channelsAfterSecond === channelsAfterFirst,
    `second call opened no new channel (${channelsAfterFirst} -> ${channelsAfterSecond})`);
const s2 = JSON.parse(String(search2.content?.[0]?.text ?? "{}"));
assert(Array.isArray(s2.results), "second paid web_search returns results");

   payment demanded for web_search (7 accepts):
     eip155:10 0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85 amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C amount=10000000000000000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42 amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913 amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85 amount=10000000000000000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x808456652fdb597867f38412077A9182bf77359F amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x036CbD53842c5426634e7929541eC2318f3dCF7e amount=10000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


TypeError: reqs.find is not a function

## `llm_complete` — the usage-derived settlement check (spike question #5)

The server verifies against the advertised per-message ceiling (what the voucher signs) but
claims the real, usage-derived amount (`withUsageDerivedSettlement` in `mcp_tools.ts`). Two
client facts shape how to read the result: the wrapper retries exactly **once**, and the paid
receipt is the **top-level `paymentResponse` field** (not `_meta`). If the server refused the
voucher, the content is the *second* payment demand, with the rejection reason in its `error`
field — a cumulative desync (the documented localStorage-channel pitfall) heals on the next
call via the corrective flow. Compare the claim against the completion's `usage` with the
Mistral rate card ($0.50/M input, $1.50/M output, 6 decimals for USDC — the same formula
`test/mcp_server.test.ts` pins). A small completion that settled for the full ceiling would
be a parity bug.

In [ ]:
const completion = await payer.callTool("llm_complete", {
    model: "mistral-large-latest",
    messages: [{ role: "user", content: "In one sentence, what is an x402 payment channel?" }],
});

// The evidence before any assumption: what the retry actually came back with. The paid
// receipt is the TOP-LEVEL paymentResponse field, and the wrapper retries exactly once —
// a server-side verify refusal means this content IS the second payment demand, with the
// rejection reason in its `error` field (cumulative mismatch etc. -> rerun the cell; the
// corrective flow re-syncs on the next call).
console.log("isError:", completion.isError, "| paymentMade:", completion.paymentMade);
console.log("raw content[0]:", String(completion.content?.[0]?.text ?? "(empty)").slice(0, 400));
console.log("paymentResponse:", JSON.stringify(completion.paymentResponse)?.slice(0, 220));

const llm = JSON.parse(String(completion.content?.[0]?.text ?? "{}"));
console.log(`   reply: ${llm.choices?.[0]?.message?.content}`);
console.log(`   usage: ${JSON.stringify(llm.usage)}`);
assert(llm.choices?.[0]?.message?.content, "paid llm_complete returns a completion");

// Expected claim for this usage, USDC rate card (same arithmetic as the REST tests):
// (prompt_tokens * 50n + completion_tokens * 150n) / 100n, capped at the advertised ceiling.
const u = llm.usage as { prompt_tokens: number; completion_tokens: number } | undefined;
if (u) {
    const expectedUsdc = (BigInt(u.prompt_tokens) * 50n + BigInt(u.completion_tokens) * 150n) / 100n;
    console.log(`   expected USDC claim for this usage: ${expectedUsdc} atomic (${expectedUsdc / 1_000_000n}.${(expectedUsdc % 1_000_000n).toString().padStart(6, "0")} USD)`);
    console.log("   -> compare against paymentResponse / the channel's claimed delta in the findings cell");
}

   payment demanded for llm_complete (7 accepts):
     eip155:10 0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85 amount=30000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C amount=30000000000000000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42 amount=30000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:8453 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913 amount=30000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85 amount=30000000000000000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x808456652fdb597867f38412077A9182bf77359F amount=30000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
     eip155:84532 0x036CbD53842c5426634e7929541eC2318f3dCF7e amount=30000 payTo=0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
   reply: undefined
   usage: undefined
   payment-response: undefined


Error: FAILED: paid llm_complete returns a completion

## Currency variants — pinning the accept explicitly

The default walk paid whichever accept the server lists first. This cell pins the other
coins explicitly with `paymentRequirementsSelector` — on Base the walk above already covered
EURe (Permit2, spike question #4); these cover EURC and USDC. Each new asset opens its own
channel (channels are per (buyer, receiver, network, asset)) — two more small deposits, by
the floor strategy above.

In [ ]:
async function payWithAsset(label: string, asset: `0x${string}` | undefined) {
    if (!asset) { console.log(`   ${label}: not deployed on this network — skipped`); return; }
    const c = createx402MCPClient({
        name: "mcp-x402-buyer-notebook", version: "1.0.0",
        schemes: [{ network: NETWORK, client: buyerScheme }],
        spendControls: { allowedAssets },
        paymentRequirementsSelector: (reqs) =>
            reqs.find((r) => (r.asset ?? "").toLowerCase() === asset.toLowerCase()),
    });
    await c.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));
    const r = await c.callTool("web_search", { q: `${label} payment over MCP` });
    const parsed = JSON.parse(String(r.content?.[0]?.text ?? "{}"));
    assert(Array.isArray(parsed.results), `${label} variant paid and returned results`);
    console.log(`   ${label} payment-response: ${JSON.stringify((r as { _meta?: Record<string, unknown> })._meta?.["x402/payment-response"])?.slice(0, 200)}`);
    await c.close();
}

await payWithAsset("EURC", config.eurcAddress);
await payWithAsset("USDC", config.usdcAddress);
console.log(`channel contexts after variants: ${channelKeyCount()}`);

## Teardown

`close()` sends a `DELETE /mcp` — which a stateless server answers with 405. The SDK must
tolerate that silently (spike question #1, the DELETE half). If teardown hangs or throws,
that is a finding, not a reason to skip it.

In [ ]:
await observer.close();
await payer.close();
console.log("closed both clients (DELETE answered 405 and the SDK tolerated it)");

## Findings

Fill in after a real run (testnet first, then flip `USE_DEPLOYED` for the gateway check):

- [ ] **Stateless server tolerated** — `initialize` + `listTools` + teardown all passed against
      `http://127.0.0.1:8086/mcp` with no session, GET/DELETE answered 405
- [ ] **Payment convention verified** — the official client read the demands out of the
      unpaid result and paid with zero custom code (`-32042` client-side, `_meta` both ways)
- [ ] **Channel deposit over MCP works** — first paid call deposited a real channel on Base Sepolia
- [ ] **EURe Permit2 over MCP works** — the default Base walk paid EURe (permit carried in the
      challenge extensions); if this failed, the fallback is dropping the EURe accept from
      `mcp_tools.ts` only
- [ ] **Usage-derived settlement parity** — `llm_complete` claimed the expected USDC amount for
      its `usage`, below the advertised ceiling
- [ ] **Gateway (deployed run)** — same walk passed against `https://mcp-agent.fretchen.eu/mcp`
      (202/405/POST semantics through Scaleway's Envoy)

Channel reuse: `channelsBefore -> after first call -> after second -> after variants`:
`___ -> ___ -> ___ -> ___` (expected: one new channel per asset paid, zero on reuse).